In [1]:
import pandas as pd
import glob

file = glob.glob("dataraw/*migr_resvalid*")[0]
print("File trovato:", file)

df = pd.read_csv(file)
print("Righe e colonne:", df.shape)
df.head()

File trovato: dataraw\migr_resvalid__custom_22933208_linear.csv.gz
Righe e colonne: (3344, 12)


,DATAFLOW,LAST UPDATE,freq,duration,reason,unit,citizen,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:MIGR_RESVALID(1.0),28/09/26 11:00:00,Annual,Total,Education reasons,Person,Albania,Austria,2009,190.0,NaN,NaN
1,ESTAT:MIGR_RESVALID(1.0),28/09/26 11:00:00,Annual,Total,Education reasons,Person,Albania,Austria,2010,228.0,NaN,NaN
2,ESTAT:MIGR_RESVALID(1.0),28/09/26 11:00:00,Annual,Total,Education reasons,Person,Albania,Austria,2011,250.0,NaN,NaN
3,ESTAT:MIGR_RESVALID(1.0),28/09/26 11:00:00,Annual,Total,Education reasons,Person,Albania,Austria,2012,265.0,NaN,NaN
4,ESTAT:MIGR_RESVALID(1.0),28/09/26 11:00:00,Annual,Total,Education reasons,Person,Albania,Austria,2013,279.0,NaN,NaN


In [2]:
df = df[["geo", "TIME_PERIOD", "reason", "OBS_VALUE", "OBS_FLAG"]]
df = df.rename(columns={
    "geo": "paese",
    "TIME_PERIOD": "anno",
    "reason": "motivo",
    "OBS_VALUE": "permessi_validi",
    "OBS_FLAG": "flag"
})
df.head()

,paese,anno,motivo,permessi_validi,flag
0,Austria,2009,Education reasons,190.0,NaN
1,Austria,2010,Education reasons,228.0,NaN
2,Austria,2011,Education reasons,250.0,NaN
3,Austria,2012,Education reasons,265.0,NaN
4,Austria,2013,Education reasons,279.0,NaN


In [3]:
traduzione_motivi = {
    "Education reasons": "Studio",
    "Employment reasons": "Lavoro",
    "Family reasons": "Famiglia",
    "Other reason": "Altro",
    "Refugee status": "Rifugiato",
    "Subsidiary protection status": "Protezione sussidiaria",
    "Total": "Totale"
}
df["motivo"] = df["motivo"].replace(traduzione_motivi)

df["motivo"].value_counts()

motivo
Totale                    570
Famiglia                  563
Altro                     549
Studio                    537
Lavoro                    525
Rifugiato                 314
Protezione sussidiaria    286
Name: count, dtype: int64

In [4]:
print("Righe prima:", len(df))
df = df[~df["paese"].str.contains("European Union|Euro area")]
print("Righe dopo:", len(df))
print("Numero di paesi:", df["paese"].nunique())

Righe prima: 3344
Righe dopo: 3270
Numero di paesi: 33


In [5]:
print("Anni: dal", df["anno"].min(), "al", df["anno"].max())
print("Valori mancanti:", df["permessi_validi"].isna().sum())
df["flag"].value_counts(dropna=False)

Anni: dal 2008 al 2025
Valori mancanti: 14


flag
NaN    3155
p        35
u        35
b        27
d        13
ep        5
Name: count, dtype: int64

In [6]:
ultimo = df["anno"].max()
df[(df["anno"] == ultimo) & (df["motivo"] == "Totale")].sort_values("permessi_validi", ascending=False).head()

,paese,anno,motivo,permessi_validi,flag
3107,Italy,2025,Totale,407423.0,NaN
2949,Greece,2025,Totale,303440.0,NaN
2895,Germany,2025,Totale,97961.0,NaN
3022,France,2025,Totale,24811.0,NaN
2809,Belgium,2025,Totale,8032.0,NaN


In [7]:
df.to_csv("dataclean/permessi_validi_albanesi.csv", index=False)
print("File salvato!", len(df), "righe")

File salvato! 3270 righe
